In [ ]:
from src.constants import DATA_PROCESSED, BRONZE_PROFILES_TABLE

profiles_table = BRONZE_PROFILES_TABLE
output_dir = str(DATA_PROCESSED)

In [ ]:
from src.utils import load_env

load_env()
import json
import numpy as np
import pandas as pd
import hashlib
import mlflow

from sklearn.decomposition import PCA
from src.db.training import to_dataframe
from src.models.similarity import embed_bio_summaries

In [ ]:
# ── Bio embeddings: derived per-player artifact, kept out of the DB ──
# Sourced from bronze.player_profiles.summary; embedded to 384 dims, then
# PCA-reduced to 10 dims (bio_0..bio_9). Saved as an independent
# player_id -> bio_* frame consumed by the NN (02_tune_nn) for the
# canonical player and opponent sides of each match row.

print("Computing bio embeddings from bronze.player_profiles.summary...")

profiles = to_dataframe(f"SELECT player_id, summary FROM {profiles_table}")
bio_full = embed_bio_summaries(profiles)

# Static PCA reduction at artifact build time; consumers read the bio
# dimension dynamically from the npz/cols, so no PCA artifact is saved.
pca = PCA(n_components=10)
bio_pca = pca.fit_transform(bio_full.drop(columns=["player_id"]))
bio_df = pd.DataFrame(bio_pca, columns=[f"bio_{i}" for i in range(10)])
bio_df.insert(0, "player_id", bio_full["player_id"].to_numpy())
bio_feat_cols = [c for c in bio_df.columns if c != "player_id"]
np.savez_compressed(
    f"{output_dir}/bio_embeddings.npz",
    player_ids=bio_df["player_id"].to_numpy(),
    vectors=bio_df[bio_feat_cols].to_numpy(np.float32),
)

with open(f"{output_dir}/bio_feature_cols.json", "w") as f:
    json.dump(bio_feat_cols, f)

print(f"Saved {len(bio_df)} player embeddings, dim {len(bio_feat_cols)}")

In [ ]:
# ── Log embeddings immutably: run URI + content hash pin them downstream ──


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read())

    return h.hexdigest()


embeddings_path = f"{output_dir}/bio_embeddings.npz"
bio_cols_path = f"{output_dir}/bio_feature_cols.json"

mlflow.set_experiment("bio-embeddings")
mlflow.set_experiment_tag("pipeline", "tune")
aux_pins: dict[str, str] = {}

with mlflow.start_run(run_name="build-bio-embeddings", tags={"pipeline": "tune"}):
    mlflow.log_artifact(embeddings_path)
    mlflow.log_artifact(bio_cols_path)
    run = mlflow.active_run()
    assert run is not None
    run_id = run.info.run_id
    aux_pins = {
        "embeddings_uri": f"runs:/{run_id}/bio_embeddings.npz",
        "embeddings_hash": _sha256(embeddings_path),
        "bio_feature_cols_uri": f"runs:/{run_id}/bio_feature_cols.json",
        "bio_feature_cols_hash": _sha256(bio_cols_path),
    }

with open(f"{output_dir}/aux_pins.json", "w") as f:
    json.dump(aux_pins, f, indent=2)

print(f"Embeddings pinned: {aux_pins['embeddings_uri']} ({aux_pins['embeddings_hash'][:12]}...)")